In [2]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
DATA_PATH = "data/pubmedqa_clean.csv"

df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["medical_text"])

print(df, df.shape)
df.head()

     question_id  ...                                       medical_text
0       21645374  ...  Question: Do mitochondria play a role in remod...
1       16418930  ...  Question: Landolt C and snellen e acuity: diff...
2        9488747  ...  Question: Syncope during bathing in infants, a...
3       17208539  ...  Question: Are the long-term results of the tra...
4       10808977  ...  Question: Can tailored interventions increase ...
..           ...  ...                                                ...
995      8921484  ...  Question: Does gestational age misclassificati...
996     16564683  ...  Question: Is there any interest to perform ult...
997     23147106  ...  Question: Is peak concentration needed in ther...
998     21550158  ...  Question: Can autologous platelet-rich plasma ...
999     17559449  ...  Question: Are sugars-free medicines more erosi...

[1000 rows x 11 columns] (1000, 11)


,question_id,question,contexts,labels,meshes,year,reasoning_required,final_decision,long_answer,context_text,medical_text
0,21645374,Do mitochondria play a role in remodelling lac...,['Programmed cell death (PCD) is the regulated...,"['BACKGROUND', 'RESULTS']","['Alismataceae', 'Apoptosis', 'Cell Differenti...",2011.0,yes,yes,Results depicted mitochondrial dynamics in viv...,Programmed cell death (PCD) is the regulated d...,Question: Do mitochondria play a role in remod...
1,16418930,Landolt C and snellen e acuity: differences in...,['Assessment of visual acuity depends on the o...,"['BACKGROUND', 'PATIENTS AND METHODS', 'RESULTS']","['Adolescent', 'Adult', 'Aged', 'Aged, 80 and ...",2006.0,no,no,"Using the charts described, there was only a s...",Assessment of visual acuity depends on the opt...,Question: Landolt C and snellen e acuity: diff...
2,9488747,"Syncope during bathing in infants, a pediatric...",['Apparent life-threatening events in infants ...,"['BACKGROUND', 'CASE REPORTS']","['Baths', 'Histamine', 'Humans', 'Infant', 'Sy...",1997.0,yes,yes,"""Aquagenic maladies"" could be a pediatric form...",Apparent life-threatening events in infants ar...,"Question: Syncope during bathing in infants, a..."
3,17208539,Are the long-term results of the transanal pul...,['The transanal endorectal pull-through (TERPT...,"['PURPOSE', 'METHODS', 'RESULTS']","['Child', 'Child, Preschool', 'Colectomy', 'Fe...",2007.0,yes,no,Our long-term study showed significantly bette...,The transanal endorectal pull-through (TERPT) ...,Question: Are the long-term results of the tra...
4,10808977,Can tailored interventions increase mammograph...,['Telephone counseling and tailored print comm...,"['BACKGROUND', 'DESIGN', 'PARTICIPANTS', 'INTE...","['Cost-Benefit Analysis', 'Female', 'Health Ma...",2000.0,yes,yes,The effects of the intervention were most pron...,Telephone counseling and tailored print commun...,Question: Can tailored interventions increase ...


In [7]:
vectorizer = TfidfVectorizer(
    stop_words = "english",
    max_features = 10000
)

medical_vectors = vectorizer.fit_transform(df["medical_text"])
print(medical_vectors.shape)

(1000, 10000)


In [13]:
def medical_records(query, top_n = 5):
    if not query.strip():
        return "Please enter a valid query."
    
    query_vector = vectorizer.transform([query])
    similarity_scores = cosine_similarity(query_vector, medical_vectors).flatten()
    top_indices = similarity_scores.argsort()[::-1][:top_n]
    
    results = df.iloc[top_indices].copy()
    results["similarity_score"] = similarity_scores[top_indices]
    
    return results[
        [
            "question",
            "context_text",
            "long_answer",
            "final_decision",
            "similarity_score" 
        ]
    ]

In [14]:
medical_records("Does aspirin reduce cardiovascular risk?", top_n = 5)

,question,context_text,long_answer,final_decision,similarity_score
686,Can common carotid intima media thickness serv...,It is not known whether common carotid intima ...,Our findings support CIMT as a significant ind...,yes,0.201875
929,"Do ""America's Best Hospitals"" perform better f...","""America's Best Hospitals,"" an influential lis...",Admission to a hospital ranked high on the lis...,yes,0.152177
392,Is high-sensitivity C-reactive protein associa...,There is a positive association between chroni...,Both hsCRP levels and the carotid IMT were str...,no,0.132539
19,Cardiovascular risk in a rural adult West Afri...,Elevated resting heart rate (RHR) is a neglect...,Significant associations were observed between...,yes,0.128881
297,Is cardiovascular evaluation necessary prior t...,Although consensus guidelines for pretreatment...,Pretreatment ECG is of limited value for patie...,no,0.114844


In [15]:
medical_records("What are the side effects of statins?", top_n = 5)

,question,context_text,long_answer,final_decision,similarity_score
52,Does pretreatment with statins improve clinica...,"In primary and secondary prevention trials, st...",The statistical power of this case-referent st...,no,0.642612
339,Do preoperative statins reduce atrial fibrilla...,Recent studies have demonstrated that statins ...,Our study indicated that preoperative statin t...,yes,0.207196
114,Are physicians aware of the side effects of an...,Angiotensin-converting enzyme inhibitors (ACE-...,"Overall, there was a poor knowledge of the sid...",no,0.086183
571,Adults with mild intellectual disabilities: ca...,Adults with a mild intellectual disability (ID...,The presently used intervention programme prov...,yes,0.078033
982,Intravenous administration of metoclopramide b...,To determine the therapeutic effect (alleviati...,Slowing the infusion rate of metoclopramide is...,yes,0.068454
